# 예제 09. 금융 업무보고서 · 공문 작성 에이전트
### 금융데이터 분석 결과를 실무 문서로 변환 · 2일 차 · 55분

> **실행 방법**: 위에서부터 순서대로 실행하세요. 셀을 클릭하고 `Shift + Enter`. AI 답변이 나오는 셀은 몇 초~수십 초 걸립니다. 셀 왼쪽이 `[*]`이면 아직 실행 중입니다.

| 항목 | 내용 |
|---|---|
| 데이터 | `data/bank_customers.csv` + 보고서 양식 `templates/report_template.docx` |
| 오늘의 결과물 | `outputs/정기예금_가입률_향상_전략보고서.docx` · `outputs/내부기안문_초안.docx` |

**오늘 해결할 질문**: "분석 결과를 위에 올릴 보고서와 기안문 초안으로 바로 만들 수 있을까?"

## 0. 실습 준비

FinanceAgent를 불러오고, 이 PC의 로컬 AI(Ollama)가 켜져 있는지 확인합니다.
`AI 연결 확인: qwen3:4b-instruct 준비 완료`가 나오면 성공입니다.

In [ ]:
import json, urllib.request
from pathlib import Path
from finance_agent import FinanceAgent
import llm_config

DATA = "data/bank_customers.csv"
models = [m["name"] for m in json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]]
print("데이터 파일:", Path(DATA).is_file())
print("AI 연결 확인:", llm_config.OLLAMA_MODEL, "준비 완료" if llm_config.OLLAMA_MODEL in models else "→ 모델이 없습니다. 강사에게 알려 주세요")
agent = FinanceAgent(provider="ollama")

## 09-1. 업무보고서는 누가 무엇을 쓰나?

| 보고서 부분 | 작성 | 이유 |
|---|---|---|
| 제목 · 작성일 · 분석 목적 | Python (고정 문구) | 매번 같아야 함 |
| 전체 고객수 · 가입률 문장, 통계표, 차트 | **Python** | 숫자는 틀리면 안 됨 |
| 주요 현황 · 검토 의견 · 향후 추진 방향 | **AI 초안** | 문장 작성은 AI가 빠름 |
| 유의사항 · 숫자 검증 결과 | Python (고정 문구 + 자동 검사) | 한계를 반드시 적음 |

## 09-2. AI에게 주기 전에 Python이 뽑은 핵심

고객군 기준마다 가입률이 가장 높은 · 낮은 고객군을 Python이 직접 찾아 AI에게 넘깁니다. AI가 표에서 잘못 읽는 일을 줄여 줍니다.

In [ ]:
from exercises import ex09_report

요약 = ex09_report.run_function(DATA, "보고서 작성")
요약["고객군별 최고·최저"]

## 09-3. 업무보고서 만들기 ⭐

In [ ]:
ex09_report.DOC_TYPE = "보고서"
결과 = agent.run("ex09", DATA, "정기예금 가입률 향상을 위한 고객관리 방안 보고서를 작성해 주세요.")

> 💬 AI 답변의 문장은 PC와 모델에 따라 조금씩 다릅니다. 표·숫자·파일은 Python이 계산하므로 모두 같게 나옵니다.

파일 탐색기에서 `outputs/정기예금_가입률_향상_전략보고서.docx`를 열어 봅니다. 표와 차트가 들어간 자리, AI가 쓴 자리를 구분해 보세요.

## 09-4. AI 문장 검토 체크리스트 ⚠️

보고서 맨 아래 **숫자 검증** 결과와 함께 아래를 확인합니다.

- [ ] 수치가 표·차트와 일치하는가?
- [ ] "반드시", "확실히" 같은 단정 표현은 없는가?
- [ ] 연령·직업만으로 특정 고객을 고르거나 배제하는 표현은 없는가?
- [ ] 데이터 출처와 한계(해외 공개 데이터, 교육용)가 적혀 있는가?

In [ ]:
from docx import Document

문서 = Document("outputs/정기예금_가입률_향상_전략보고서.docx")
for p in 문서.paragraphs:
    if "가입률은" in p.text or "숫자 검증" in p.text:
        print("·", p.text)

## 09-5. 같은 분석으로 내부 기안문 만들기

문서 종류만 바꾸면 같은 분석 결과가 **기안문 형식**(수신 · 제목 · 본문 · 붙임 · 결재란)으로 나옵니다.

In [ ]:
ex09_report.DOC_TYPE = "기안문"
결과 = agent.run("ex09", DATA, "정기예금 가입률 향상 방안 검토를 내부 기안문으로 작성해 주세요.")
ex09_report.DOC_TYPE = "보고서"

## 09-6. [선택] 🔧 보고서 관점 바꾸기

업무 요청을 바꾸면 AI가 쓰는 검토 의견과 추진 방향이 달라집니다. 표와 차트는 그대로입니다.

In [ ]:
결과 = agent.run("ex09", DATA, "대출이 없는 고객과 잔액이 많은 고객을 중심으로 정기예금 안내 방안을 검토해 주세요.")     # ← 바꿔 보세요

## 09-7. [선택] 우리 회사 보고서 양식 쓰기

`templates/report_template.docx`를 회사 양식으로 바꿔 두면 글꼴 · 머리글 · 제목 스타일이 그대로 적용됩니다.
단, 양식 파일 안에 Word 기본 스타일(제목, 제목 1, 목록 글머리 기호, 캡션)이 있어야 합니다. 원본 양식은 꼭 따로 보관하세요.

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| 핵심 최고·최저 미리 보기 | `ex09_report.run_function(DATA, 질문)["고객군별 최고·최저"]` |
| 업무보고서 Word | `ex09_report.DOC_TYPE = "보고서"` → `agent.run("ex09", DATA, 질문)` |
| 내부 기안문 Word | `ex09_report.DOC_TYPE = "기안문"` → `agent.run("ex09", DATA, 질문)` |
| Word 내용 읽기 | `Document("파일.docx").paragraphs` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션.** 09-4의 코드를 이용해 보고서 안에서 "전체 고객"이 들어간 문장을 찾아 출력해 보세요.
**이렇게 나오면 성공!** `· 전체 고객 45,211명 중 5,289명이 정기예금에 가입하여 전체 가입률은 11.7%임.`

**다음 예제 전에 이것만은 꼭!**
- [ ] 숫자·표·차트는 Python, 문장은 AI 초안, 최종 확인은 사람
- [ ] 같은 분석 → 보고서 / 기안문처럼 형식만 바꿔 재사용

In [ ]:
# 미션 — 09-4 셀과 같은 모양입니다
문서 = Document("outputs/정기예금_가입률_향상_전략보고서.docx")
for p in 문서.paragraphs:
    if "숫자 검증" in p.text:      # ← "전체 고객"으로
        print("·", p.text)

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
문서 = Document("outputs/정기예금_가입률_향상_전략보고서.docx")
for p in 문서.paragraphs:
    if "전체 고객" in p.text:
        print("·", p.text)

---
### ▶ 다음 예제 10. 금융업무 통합 자동화 에이전트
지금까지 따로 실행한 Excel · 차트 · 보고서를 **한 번의 실행**으로 모두 만듭니다.